In [0]:
%sql
WITH base_metrics AS (
    SELECT
        Market,
        Order_Region,
        Product_Category_Id,
        Days_for_shipping_real AS actual_lead_time,
        Days_for_shipment_scheduled AS scheduled_lead_time,
        (Days_for_shipping_real - Days_for_shipment_scheduled) AS lead_time_delay,
        CASE 
            WHEN Late_delivery_risk = 0 THEN 1 
            ELSE 0
        END AS on_time
    FROM v_supply_chain_performance
),

    route_stats as (
        SELECT
            Market,
            Order_Region,
            Product_Category_Id,
            AVG(on_time) OVER(PARTITION BY Market, Order_Region, Product_Category_Id) * 100 AS ontime_rate_pct,
        AVG(actual_lead_time) OVER(PARTITION BY Market, Order_Region, Product_Category_Id) AS avg_lead_time,
        VAR_SAMP(actual_lead_time) OVER(PARTITION BY Market, Order_Region, Product_Category_Id) AS lead_time_variance
            FROM base_metrics
    )

SELECT DISTINCT 
    Market,
    Order_Region,
    Product_Category_Id,
    ROUND(ontime_rate_pct, 2) AS ontime_rate_pct,
    ROUND(avg_lead_time, 2) AS avg_lead_time_days,
    ROUND(lead_time_variance, 2) AS lead_time_variance
FROM route_stats
ORDER BY lead_time_variance DESC
LIMIT 10;